# TinyVAD Student Model

This cell defines the TinyVAD student network used for knowledge distillation. It includes the CNN, GRU, and output layers, and prints the parameter count so you can verify that the student model is much smaller than the teacher.

In [1]:
import torch
import torch.nn as nn

class TinyVAD(nn.Module):
    """
    Small student model for knowledge distillation
    Target: ~20K parameters (5x smaller than Teacher)
    Structure: CNN -> GRU -> Linear
    """
    def __init__(self):
        super().__init__()
        
        # Small CNN
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.LeakyReLU(0.01),
            nn.MaxPool2d(kernel_size=(1, 2)),
            nn.Dropout2d(0.1)
        )
        
        # Small GRU (unidirectional, smaller hidden size)
        self.gru = nn.GRU(
            input_size=160,
            hidden_size=16,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )
        
        # Output
        self.output = nn.Sequential(
            nn.Linear(16, 8),
            nn.LeakyReLU(0.01),
            nn.Linear(8, 1)
        )
    
    def forward(self, x):
        # x: (batch, time, features)
        B, T, F = x.shape
        
        # CNN expects (batch, channel, time, freq)
        x = x.unsqueeze(1)           # (B, 1, T, F)
        x = self.cnn(x)              # (B, 16, T, F//4)
        
        # Reshape for GRU
        B, C, T, F = x.shape
        x = x.permute(0, 2, 1, 3)   # (B, T, C, F)
        x = x.reshape(B, T, C*F)    # (B, T, C*F)
        
        # GRU
        x, _ = self.gru(x)          # (B, T, 16)
        
        # Output
        x = self.output(x)          # (B, T, 1)
        x = torch.sigmoid(x)
        return x.squeeze(-1)         # (B, T)


# Check parameter count
student = TinyVAD()
n_params = sum(p.numel() for p in student.parameters())
print(f"TinyVAD parameters: {n_params:,}")
print(f"Teacher parameters: 109,700")
print(f"Compression ratio:  {109700/n_params:.1f}x smaller")

TinyVAD parameters: 9,937
Teacher parameters: 109,700
Compression ratio:  11.0x smaller


# Run Distillation

This cell launches `distill.py`, which starts training the TinyVAD student model and writes the results to the notebook output or the training log.

In [10]:
!python distill.py

Device: cuda
Loaded 11180 examples from /courses/CS6140.202630/students/cong.m/results/VAD/save/train.json
Loaded 1932 examples from /courses/CS6140.202630/students/cong.m/results/VAD/save/valid.json
torchvision is not available - cannot save figures
Teacher loaded!
Student parameters: 9,937
Epoch   1/30 | Loss: 0.5606 | F1: 0.8655 | Prec: 0.8003 | Rec: 0.9423
  --> Best model saved! F1=0.8655
Epoch   2/30 | Loss: 0.5035 | F1: 0.9038 | Prec: 0.8950 | Rec: 0.9127
  --> Best model saved! F1=0.9038
Epoch   3/30 | Loss: 0.4889 | F1: 0.9050 | Prec: 0.8903 | Rec: 0.9201
  --> Best model saved! F1=0.9050
Epoch   4/30 | Loss: 0.4818 | F1: 0.9099 | Prec: 0.8935 | Rec: 0.9270
  --> Best model saved! F1=0.9099
Epoch   5/30 | Loss: 0.4782 | F1: 0.9133 | Prec: 0.9107 | Rec: 0.9159
  --> Best model saved! F1=0.9133
Epoch   6/30 | Loss: 0.4755 | F1: 0.9102 | Prec: 0.8979 | Rec: 0.9229
Epoch   7/30 | Loss: 0.4733 | F1: 0.9122 | Prec: 0.9052 | Rec: 0.9193
Epoch   8/30 | Loss: 0.4710 | F1: 0.9141 | Prec

# Check Training Progress

This cell uses `grep` to show the `Epoch` lines from the training log, so you can quickly confirm that distillation is progressing normally.

In [ ]:
# Check training progress
!grep "Epoch" /courses/CS6140.202630/students/cong.m/distill_log.txt